In [8]:
# twinkling_stars_gif.py  (4K large dimensions + ultra-gentle long cycle)
# Requirements: pip install pillow numpy

import numpy as np
from PIL import Image

# ────────────────────────────────────────────────
#  SETTINGS – large 4K dimensions, still very gentle
# ────────────────────────────────────────────────
WIDTH, HEIGHT = 3840, 2160          # much larger (4K) for better quality / coverage
NUM_STARS = 220                     # scaled up slightly to maintain density on larger canvas
NUM_FRAMES = 960                    # ~2 minutes @ 8 fps → long gentle seamless loop
FPS = 8                             # peaceful playback
BASE_COLOR = (245, 242, 235)        # creamy off-white
MIN_BRIGHTNESS = 0
MAX_BRIGHTNESS = 24                 # very mild darkness variation
GLOW = True
SEED = 42

np.random.seed(SEED)

# ────────────────────────────────────────────────
#  Star positions & strengths (float for smooth zoom)
# ────────────────────────────────────────────────
stars_x = np.random.uniform(0, WIDTH, NUM_STARS).astype(float)
stars_y = np.random.uniform(0, HEIGHT, NUM_STARS).astype(float)
base_strength = np.random.uniform(0.7, 1.0, NUM_STARS)

# ────────────────────────────────────────────────
#  Generate frames
# ────────────────────────────────────────────────
frames = []

for frame_idx in range(NUM_FRAMES):
    # Extremely slow phase (full cycle ~120 seconds)
    phase = frame_idx / NUM_FRAMES * 2 * np.pi
    phase_offsets = np.random.uniform(0, 2*np.pi, NUM_STARS)
    
    # Very mild twinkling variation
    multipliers = 0.55 + 0.45 * (np.sin(phase + phase_offsets + frame_idx * 0.0065) + 1) / 2
    multipliers = np.clip(multipliers, 0.0, 1.0)
    
    # Gentle global zoom (±1.5%, very slow)
    zoom_phase = frame_idx * 0.0085
    zoom_factor = 1.0 + 0.015 * np.sin(zoom_phase)
    
    cx, cy = WIDTH / 2.0, HEIGHT / 2.0
    zoomed_x = cx + (stars_x - cx) * zoom_factor
    zoomed_y = cy + (stars_y - cy) * zoom_factor
    
    frame_data = np.full((HEIGHT, WIDTH, 3), BASE_COLOR, dtype=np.uint8)
    
    for i in range(NUM_STARS):
        x = int(zoomed_x[i])
        y = int(zoomed_y[i])
        
        if not (0 <= x < WIDTH and 0 <= y < HEIGHT):
            continue
        
        darkness = int(MAX_BRIGHTNESS * base_strength[i] * multipliers[i])
        bright = MIN_BRIGHTNESS + darkness
        
        # Center dot
        frame_data[y, x] = [bright, bright, bright]
        
        # Very faint halo
        if GLOW and darkness > 6:
            glow = darkness // 10
            for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
                ny, nx = y + dy, x + dx
                if 0 <= ny < HEIGHT and 0 <= nx < WIDTH:
                    current = frame_data[ny, nx]
                    frame_data[ny, nx] = np.minimum(current, [glow, glow, glow])
    
    img = Image.fromarray(frame_data, mode='RGB')
    frames.append(img)

# ────────────────────────────────────────────────
#  Save
# ────────────────────────────────────────────────
frames[0].save(
    "stars-twinkle-cream-4k-gentle.gif",
    save_all=True,
    append_images=frames[1:],
    duration=1000 // FPS,
    loop=0,
    optimize=True,
    disposal=2
)

print("Generated stars-twinkle-cream-4k-gentle.gif (3840×2160)")

Generated stars-twinkle-cream-4k-gentle.gif (3840×2160)


In [1]:
import numpy as np
from PIL import Image
import colorsys

# ────────────────────────────────────────────────
#  SETTINGS
# ────────────────────────────────────────────────
WIDTH, HEIGHT = 3840, 2160       # 4K
NUM_STARS   = 220
NUM_FRAMES  = 300                # smooth loop
FPS         = 12
BASE_COLOR  = (245, 242, 235)    # creamy off-white
GLOW        = True
SEED        = 42

np.random.seed(SEED)

# ────────────────────────────────────────────────
#  Star properties
# ────────────────────────────────────────────────
stars_x = np.random.uniform(0, WIDTH, NUM_STARS)
stars_y = np.random.uniform(0, HEIGHT, NUM_STARS)
base_strength = np.random.uniform(0.68, 1.0, NUM_STARS)

phase_offsets = np.random.uniform(0, 2 * np.pi, NUM_STARS)

star_hues = np.random.uniform(0, 1, NUM_STARS)
star_colors = [
    tuple(int(255 * c) for c in colorsys.hsv_to_rgb(h, 1.0, 1.0))
    for h in star_hues
]

# ────────────────────────────────────────────────
#  Background once
# ────────────────────────────────────────────────
background = np.full((HEIGHT, WIDTH, 3), BASE_COLOR, dtype=np.uint8)

# ────────────────────────────────────────────────
#  Generate frames
# ────────────────────────────────────────────────
frames = []
BLACK = np.array([0., 0., 0.])

for frame_idx in range(NUM_FRAMES):
    phase = frame_idx * (2 * np.pi / NUM_FRAMES) * 1.6
    multipliers = (np.sin(phase + phase_offsets) + 1) / 2
    multipliers = np.clip(multipliers, 0.0, 1.0)

    zoom_phase = frame_idx * 0.016
    zoom_factor = 1.0 + 0.018 * np.sin(zoom_phase)

    cx, cy = WIDTH / 2.0, HEIGHT / 2.0
    zoomed_x = cx + (stars_x - cx) * zoom_factor
    zoomed_y = cy + (stars_y - cy) * zoom_factor

    frame_data = background.copy()

    for i in range(NUM_STARS):
        x = int(zoomed_x[i])
        y = int(zoomed_y[i])

        if not (0 <= x < WIDTH and 0 <= y < HEIGHT):
            continue

        intensity = multipliers[i] * base_strength[i]
        center_alpha = intensity * 0.92          # stronger center

        # ─── Much larger black core (~9×9 with falloff) ───
        for dy in range(-4, 5):
            for dx in range(-4, 5):
                dist = max(abs(dy), abs(dx))          # Chebyshev
                if dist > 4:
                    continue

                if dist == 0:
                    alpha_here = center_alpha * 1.00
                elif dist == 1:
                    alpha_here = center_alpha * 0.92
                elif dist == 2:
                    alpha_here = center_alpha * 0.70
                elif dist == 3:
                    alpha_here = center_alpha * 0.38
                elif dist == 4:
                    alpha_here = center_alpha * 0.14
                else:
                    continue

                py = y + dy
                px = x + dx
                if 0 <= py < HEIGHT and 0 <= px < WIDTH:
                    current = frame_data[py, px].astype(float)
                    blended = (1 - alpha_here) * current + alpha_here * BLACK
                    frame_data[py, px] = np.clip(blended, 0, 255).astype(np.uint8)

        # ─── Rainbow glow – larger inner + outer ───
        if GLOW and intensity > 0.10:
            glow_color = np.array(star_colors[i], dtype=float)

            # Inner glow (~ ±3 px)
            glow_alpha_inner = intensity * 0.72
            for dy in range(-3, 4):
                for dx in range(-3, 4):
                    if max(abs(dy), abs(dx)) <= 1:   # center + very close already dark
                        continue
                    if max(abs(dy), abs(dx)) > 3:
                        continue
                    ny = y + dy
                    nx = x + dx
                    if 0 <= ny < HEIGHT and 0 <= nx < WIDTH:
                        current = frame_data[ny, nx].astype(float)
                        new_c = (1 - glow_alpha_inner) * current + glow_alpha_inner * glow_color
                        frame_data[ny, nx] = np.clip(new_c, 0, 255).astype(np.uint8)

            # Outer glow (~ ±5–6 px, softer)
            glow_alpha_outer = intensity * 0.36
            for dy in range(-6, 7):
                for dx in range(-6, 7):
                    cheb = max(abs(dy), abs(dx))
                    if cheb <= 3:     # already handled
                        continue
                    if cheb > 6:
                        continue
                    # fade with distance
                    falloff = 1.0 - (cheb - 3) / 3.5
                    alpha_here = glow_alpha_outer * max(0, falloff)

                    ny = y + dy
                    nx = x + dx
                    if 0 <= ny < HEIGHT and 0 <= nx < WIDTH and alpha_here > 0.03:
                        current = frame_data[ny, nx].astype(float)
                        new_c = (1 - alpha_here) * current + alpha_here * glow_color
                        frame_data[ny, nx] = np.clip(new_c, 0, 255).astype(np.uint8)

    img = Image.fromarray(frame_data, mode='RGB')
    frames.append(img)

# ────────────────────────────────────────────────
#  Save
# ────────────────────────────────────────────────
output_filename = "stars-twinkle-cream-4k-gentle.gif"

frames[0].save(
    output_filename,
    save_all=True,
    append_images=frames[1:],
    duration=1000 // FPS,
    loop=0,
    optimize=True,
    disposal=2
)

print(f"Generated {output_filename} ({WIDTH}×{HEIGHT}, {NUM_FRAMES} frames, {FPS} FPS)")

Generated stars-twinkle-cream-4k-gentle.gif (3840×2160, 300 frames, 12 FPS)
